In [14]:
import itertools
import re

def get_symbols(formula):
    words = re.findall(r'\b[a-zA-Z_][a-zA-Z0-9_]*\b', formula)
    operators = {'and', 'or', 'not', 'implies', 'iff', 'true', 'false', 'v'}
    return sorted(list(set(w for w in words if w.lower() not in operators)))

def evaluate_formula(formula, model):
    sorted_vars = sorted(model.keys(), key=len, reverse=True)

    expr = formula
    for var in sorted_vars:
        expr = re.sub(rf'\b{var}\b', str(model[var]), expr)

    expr = expr.replace('<->', ' == ')
    expr = expr.replace('->', ' implies ')
    expr = expr.replace('^', ' and ')
    expr = expr.replace('v', ' or ')
    expr = expr.replace('~', ' not ')

    for op in ['and', 'or', 'not', 'implies']:
        expr = re.sub(rf'\b{op}\b', op.lower(), expr, flags=re.IGNORECASE)

    eval_env = {
        'implies': lambda x, y: (not x) or y,
        'True': True,
        'False': False
    }

    pattern = r'(\bTrue\b|\bFalse\b|\([^()]*\))\s+implies\s+(\bTrue\b|\bFalse\b|\([^()]*\))'
    for _ in range(5):
        next_expr = re.sub(pattern, r'implies(\1, \2)', expr)
        if next_expr == expr:
            break
        expr = next_expr

    try:
        return bool(eval(expr, {"__builtins__": None}, eval_env))
    except Exception as e:
        raise ValueError(f"Error evaluating formula '{formula}' (processed as '{expr}'): {e}")

def check_entailment(kb_formulas, query_formula):
    all_formulas = kb_formulas + [query_formula]
    symbols = sorted(list(set().union(*(get_symbols(f) for f in all_formulas))))

    print(f"Symbols detected: {symbols}")
    print("-" * 80)
    header = " | ".join(symbols) + " || " + " | ".join([f"KB_{i+1}" for i in range(len(kb_formulas))]) + " || Query"
    print(header)
    print("-" * len(header))

    entails = True
    for combination in itertools.product([True, False], repeat=len(symbols)):
        model = dict(zip(symbols, combination))

        kb_values = [evaluate_formula(f, model) for f in kb_formulas]
        kb_true = all(kb_values)
        query_val = evaluate_formula(query_formula, model)

        model_str = " | ".join(["T" if model[sym] else "F" for sym in symbols])
        kb_str = " | ".join(["T" if val else "F" for val in kb_values])
        query_str = "T" if query_val else "F"

        row_suffix = ""
        if kb_true:
            row_suffix = " <-- KB is True"
            if not query_val:
                entails = False
                row_suffix += " (Counterexample! Query is False)"

        print(f"{model_str} || {kb_str} || {query_str}{row_suffix}")

    print("-" * len(header))
    return entails

In [17]:
print("Enter your Knowledge Base formulas (one per line, leave empty to finish).")
print("Supported operators:")
print("  Conjunction: ^  (or 'and')")
print("  Disjunction: v  (or 'or')")
print("  Negation:    ~  (or 'not')")
print("  Implication: -> (or 'implies')")
print("  Equivalence: <-> (or 'iff')")
print("Example: (P -> Q) ^ Q")
print("-" * 40)

kb_inputs = []
while True:
    inp = input("Enter KB formula (or press Enter to finish): ").strip()
    if not inp:
        break
    kb_inputs.append(inp)

query_input = input("Enter Query formula: ").strip()

if kb_inputs and query_input:
    result = check_entailment(kb_inputs, query_input)
    if result:
        print("\nRESULT: The Knowledge Base ENTAILS the Query! (KB |= Query)")
    else:
        print("\nRESULT: The Knowledge Base DOES NOT ENTAIL the Query! (KB |/= Query)")
else:
    print("Error: You must provide at least one KB formula and a Query.")

Enter your Knowledge Base formulas (one per line, leave empty to finish).
Supported operators:
  Conjunction: ^  (or 'and')
  Disjunction: v  (or 'or')
  Negation:    ~  (or 'not')
  Implication: -> (or 'implies')
  Equivalence: <-> (or 'iff')
Example: (P -> Q) ^ Q
----------------------------------------
Enter KB formula (or press Enter to finish): (P -> Q) ^ Q
Enter KB formula (or press Enter to finish): 
Enter Query formula: Q
Symbols detected: ['P', 'Q']
--------------------------------------------------------------------------------
P | Q || KB_1 || Query
----------------------
T | T || T || T <-- KB is True
T | F || F || F
F | T || T || T <-- KB is True
F | F || F || F
----------------------

RESULT: The Knowledge Base ENTAILS the Query! (KB |= Query)
